# SleepAgent Step Debug Notebook

This notebook runs SleepAgent by function, one step at a time: Data Foundation, Knowledge Grounding, Scientific Loop, and Co-Scientist Core.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'sleep_ai_scientist').exists():
    ROOT = Path('/home/jwj/code/SleepAgent')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

PosixPath('/home/jwj/code/SleepAgent')

In [2]:
from sleep_ai_scientist.foundation.foundation_pipeline import run_foundation_pipeline
from sleep_ai_scientist.grounding.grounding_pipeline import run_grounding_pipeline
from sleep_ai_scientist.hypothesis.hypothesis_pipeline import run_hypothesis_pipeline
from sleep_ai_scientist.experiment.experiment_pipeline import run_experiment_pipeline
from sleep_ai_scientist.hypothesis.co_scientist_pipeline import run_co_scientist_pipeline
from sleep_ai_scientist.common.io import read_csv, read_json, read_yaml

CONFIGS = {
    'foundation': 'configs/foundation_config.yaml',
    'grounding': 'configs/grounding_config.yaml',
    'hypothesis': 'configs/hypothesis_config.yaml',
    'experiment': 'configs/experiment_config.yaml',
    'co_scientist': 'configs/co_scientist_config.yaml',
}
CONFIGS

{'foundation': 'configs/foundation_config.yaml',
 'grounding': 'configs/grounding_config.yaml',
 'hypothesis': 'configs/hypothesis_config.yaml',
 'experiment': 'configs/experiment_config.yaml',
 'co_scientist': 'configs/co_scientist_config.yaml'}

## 1. Data Foundation

Build or validate foundation tables from configured inputs or toy fixtures.

In [ ]:
foundation_result = run_foundation_pipeline(CONFIGS['foundation'])
foundation_result

In [3]:
subject_index = read_csv(Path('data/foundation/subject_index.csv'))
feature_registry = read_csv(Path('data/foundation/feature_registry.csv'))
len(subject_index), len(feature_registry), subject_index[:2], feature_registry[:3]

(5,
 19,
 [{'subject_id': 'S001',
   'group': 'INS',
   'age': '34.0',
   'sex': 'F',
   'has_EEG': 'True',
   'has_fMRI': 'True',
   'has_DTI': 'True',
   'has_MRI': 'True',
   'has_scales': 'True',
   'available_modalities': 'EEG;fMRI;DTI;MRI;scales',
   'overall_qc_status': 'pass',
   'notes': ''},
  {'subject_id': 'S002',
   'group': 'INS',
   'age': '42.0',
   'sex': 'M',
   'has_EEG': 'True',
   'has_fMRI': 'True',
   'has_DTI': 'True',
   'has_MRI': 'True',
   'has_scales': 'True',
   'available_modalities': 'EEG;fMRI;DTI;MRI;scales',
   'overall_qc_status': 'caution',
   'notes': ''}],
 [{'feature_name': 'slow_wave_density',
   'modality': 'EEG',
   'source_file': '/home/jwj/code/SleepAgent/data/fixtures/toy_eeg_features.csv',
   'source_column': 'slow_wave_density',
   'role': 'feature',
   'dtype': 'numeric',
   'unit': '',
   'description': 'EEG variable slow_wave_density',
   'missing_rate': '0.2',
   'n_available': '4',
   'qc_dependency': "['EEG']",
   'approved': 'True',

## 2. Knowledge Grounding

Extract evidence, score evidence quality, map literature concepts to data variables, and build profiles.

In [4]:
grounding_result = run_grounding_pipeline(CONFIGS['grounding'])
grounding_result

{'papers': 3,
 'retrieval_hits': 0,
 'evidence': 14,
 'analysis_ready_variables': 19,
 'graph_nodes': 54,
 'graph_edges': 107,
 'report_path': '/home/jwj/code/SleepAgent/reports/grounding_report.md',
 'output_grounding_dir': '/home/jwj/code/SleepAgent/outputs/grounding',
 'output_profiles_dir': '/home/jwj/code/SleepAgent/outputs/profiles'}

In [5]:
evidence = read_csv(Path('outputs/grounding/evidence_table.csv'))
analysis_ready = read_yaml(Path('outputs/profiles/analysis_ready_profile.yaml'))
len(evidence), evidence[:3], len(analysis_ready.get('features', []))

(14,
 [{'evidence_id': 'evidence_f7ca755a2467',
   'paper_id': 'toy001',
   'claim': 'Insomnia and thalamocortical hyperarousal: thalamocortical coupling related to thalamus_DMN_FC',
   'population': 'insomnia',
   'modality': 'EEG-fMRI',
   'variable_or_feature': 'thalamus_DMN_FC',
   'mechanism': 'thalamocortical coupling',
   'direction': 'support',
   'evidence_type': 'empirical',
   'limitation': 'small sample.',
   'confidence_score': '0.7',
   'evidence_quality_score': '0.98'},
  {'evidence_id': 'evidence_f7ca755a2467',
   'paper_id': 'toy001',
   'claim': 'Insomnia and thalamocortical hyperarousal: thalamocortical coupling related to thalamus_DMN_FC',
   'population': 'insomnia',
   'modality': 'fMRI',
   'variable_or_feature': 'thalamus_DMN_FC',
   'mechanism': 'thalamocortical coupling',
   'direction': 'support',
   'evidence_type': 'empirical',
   'limitation': 'small sample.',
   'confidence_score': '0.7',
   'evidence_quality_score': '0.98'},
  {'evidence_id': 'evidence_f

## 3. Hypothesis Engine

Generate grounded candidate hypotheses and rank them with pre-analysis scores.

In [6]:
hypothesis_result = run_hypothesis_pipeline(CONFIGS['hypothesis'])
hypothesis_result

{'hypothesis_pool': '/home/jwj/code/SleepAgent/outputs/hypotheses/hypothesis_pool.json',
 'hypothesis_registry': '/home/jwj/code/SleepAgent/outputs/hypotheses/hypothesis_registry.csv',
 'top_k_hypotheses': '/home/jwj/code/SleepAgent/outputs/hypotheses/top_k_hypotheses.json',
 'hypothesis_lineage': '/home/jwj/code/SleepAgent/outputs/hypotheses/hypothesis_lineage.json',
 'generated_hypotheses': 7,
 'screened_hypotheses': 7,
 'top_k': 5}

In [7]:
top_k = read_json(Path('outputs/hypotheses/top_k_hypotheses.json'))
[(item['hypothesis_id'], item['title'], item.get('pre_analysis_score')) for item in top_k]

[('H006',
  'Thalamus-DMN connectivity association with insomnia severity',
  0.907),
 ('H004', 'Spindle density association with thalamocortical coupling', 0.882),
 ('H003', 'Slow-wave density association with insomnia severity', 0.8584),
 ('H005', 'Thalamic radiation FA association with slow-wave density', 0.846),
 ('H002', 'INS vs HC beta power difference', 0.84)]

## 4. Computational Experiment Loop

Create draft plans, lock plans, run simple linear models, robustness checks, critic reviews, and registry updates.

In [8]:
experiment_result = run_experiment_pipeline(CONFIGS['experiment'])
{k: v for k, v in experiment_result.items() if k not in {'results', 'robustness', 'reviews'}}

{'draft_plans': 5,
 'locked_plans': 5,
 'updates': {'registry_updates': 5, 'null_registry_updates': 2},
 'report': '/home/jwj/code/SleepAgent/reports/scientific_loop_report.md'}

In [9]:
experiment_result['results'][0], experiment_result['reviews'][0]

({'experiment_id': 'EXP_H002',
  'hypothesis_id': 'H002',
  'model_formula': 'beta_power ~ group + medication',
  'n_total': 5,
  'n_used': 4,
  'coefficients': {'Intercept': 0.205, 'group': 0.08, 'medication': -0.015},
  'p_values': {'Intercept': 0.00627,
   'group': 0.257899,
   'medication': 0.729034},
  'corrected_p_values': {'medication': 0.729034, 'group': 0.515798},
  'effect_sizes': {'group': 0.730297, 'medication': -0.223607},
  'confidence_intervals': {'Intercept': [0.058, 0.352],
   'group': [-0.058593, 0.218593],
   'medication': [-0.09987, 0.06987]},
  'model_fit': {'r_squared': 0.583333},
  'warnings': ['encoded_categorical:group', 'encoded_categorical:medication'],
  'status': 'ok'},
 {'experiment_id': 'EXP_H002',
  'hypothesis_id': 'H002',
  'decision': 'hold',
  'evidence_strength': 'weak',
  'claim_strength': 'not_supported',
  'main_concerns': ['sample_size_below_gate',
   'n_used_below_configured_minimum',
   'key_confound_review_required'],
  'required_followups': 

## 5. Co-Scientist Core

Run candidate pooling, proximity clustering, reflection, tournament ranking, evolution, and meta-review.

In [10]:
co_scientist_result = run_co_scientist_pipeline(CONFIGS['co_scientist'])
co_scientist_result

{'candidate_pool': '/home/jwj/code/SleepAgent/outputs/co_scientist/candidate_pool.json',
 'candidate_count': 7,
 'cluster_count': 9,
 'reflection_count': 12,
 'ranking_pair_count': 66,
 'evolved_count': 5,
 'lineage_updates': {'registry_rows_added': 5,
  'lineage_nodes_added': 5,
  'lineage_edges_added': 5},
 'co_scientist_top_k': '/home/jwj/code/SleepAgent/outputs/co_scientist/co_scientist_top_k.json',
 'meta_review_report': '/home/jwj/code/SleepAgent/outputs/co_scientist/meta_review_report.md',
 'co_scientist_report': '/home/jwj/code/SleepAgent/reports/co_scientist_report.md'}

In [12]:
clusters = read_json(Path('outputs/co_scientist/proximity_clusters.json'))
reviews = read_json(Path('outputs/co_scientist/reflection_reviews.json'))
co_top_k = read_json(Path('outputs/co_scientist/co_scientist_top_k.json'))
len(clusters), clusters[:2], len(reviews), [(item['hypothesis_id'], item['title']) for item in co_top_k[:5]]

(9,
 [{'cluster_id': 'C001',
   'hypothesis_ids': ['H006', 'H006_refine1'],
   'representative_id': 'H006',
   'cluster_label': 'thalamocortical coupling',
   'diversity_score': 0.175,
   'similarity_summary': '2 hypotheses around thalamocortical coupling'},
  {'cluster_id': 'C002',
   'hypothesis_ids': ['H004', 'H004_refine1'],
   'representative_id': 'H004',
   'cluster_label': 'thalamocortical coupling',
   'diversity_score': 0.219,
   'similarity_summary': '2 hypotheses around thalamocortical coupling'}],
 12,
 [('H004_mutate1',
   'Mutated symptom-association variant of Spindle density association with thalamocortical coupling'),
  ('H006_refine1',
   'Refined: Thalamus-DMN connectivity association with insomnia severity'),
  ('H006', 'Thalamus-DMN connectivity association with insomnia severity'),
  ('H004', 'Spindle density association with thalamocortical coupling'),
  ('H004_H006_merge1',
   'Merged multimodal proposal: thalamocortical coupling and thalamocortical coupling')])

## 6. Compatibility Check

`outputs/co_scientist/co_scientist_top_k.json` is compatible with the experiment pipeline input format. To run experiments on Co-Scientist proposals, point `configs/experiment_config.yaml -> inputs.top_k_hypotheses` to that file or create a separate experiment config.